In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
import numpy as np
import cv2
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical


In [ ]:
# Define paths
base_dir = '/content/drive/MyDrive/Defence'  # <-- change if different
categories = ['COVID', 'NORMAL']
IMG_SIZE = 224


In [ ]:
def preprocess_image(path):
    # Grayscale
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)

    # Resize
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

    # Normalize
    img = img / 255.0

    # Histogram Equalization
    img_eq = cv2.equalizeHist((img * 255).astype(np.uint8))

    # CLAHE
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img_clahe = clahe.apply(img_eq)

    # Denoising
    img_blur = cv2.GaussianBlur(img_clahe, (3, 3), 0)

    # Final shape for DL
    img_final = np.expand_dims(img_blur, axis=-1)  # (224,224,1)

    return img_final


In [ ]:
data = []
labels = []

for label in categories:
    folder = os.path.join(base_dir, label)
    class_num = categories.index(label)
    for filename in tqdm(os.listdir(folder), desc=label):
        try:
            path = os.path.join(folder, filename)
            img = preprocess_image(path)
            data.append(img)
            labels.append(class_num)
        except:
            continue

X = np.array(data, dtype='float32')
y = np.array(labels)


In [ ]:
# First split into 85% train+val and 15% test
X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)

# Then split trainval into 70/15
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=0.176, stratify=y_trainval, random_state=42)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")


In [ ]:
# Add channel 3 for pretrained networks
X_train_3c = np.repeat(X_train, 3, axis=-1)
X_val_3c = np.repeat(X_val, 3, axis=-1)
X_test_3c = np.repeat(X_test, 3, axis=-1)

y_train_cat = to_categorical(y_train)
y_val_cat = to_categorical(y_val)
y_test_cat = to_categorical(y_test)


In [ ]:
def build_custom_cnn():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1)),
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
        tf.keras.layers.MaxPooling2D(2, 2),
        tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
        tf.keras.layers.MaxPooling2D(2, 2),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(2, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model


In [ ]:
def load_pretrained_model(base_model_class):
    base_model = base_model_class(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base_model.trainable = False  # Freeze base
    model = tf.keras.Sequential([
        base_model,
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(2, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model


Custom CNN
VGG16
ResNet50
DenseNet121
InceotionV3
Xception

In [ ]:
from tensorflow.keras.applications import VGG16, ResNet50, DenseNet121, InceptionV3, Xception

models_list = {
    "Custom_CNN": build_custom_cnn(),
    "VGG16": load_pretrained_model(VGG16),
    "ResNet50": load_pretrained_model(ResNet50),
    "DenseNet121": load_pretrained_model(DenseNet121),
    "InceptionV3": load_pretrained_model(InceptionV3),
    "Xception": load_pretrained_model(Xception),
}

histories = {}

for name, model in models_list.items():
    print(f"\nTraining {name}...")
    if name == "Custom_CNN":
        history = model.fit(X_train, y_train_cat, validation_data=(X_val, y_val_cat), epochs=5, batch_size=32)
    else:
        history = model.fit(X_train_3c, y_train_cat, validation_data=(X_val_3c, y_val_cat), epochs=5, batch_size=32)
    histories[name] = history
    model.save(f'/content/drive/MyDrive/XrayModels/{name}.h5')


In [ ]:
def show_preprocessed_samples(X, y, title):
    plt.figure(figsize=(12, 4))
    for i in range(6):
        plt.subplot(1, 6, i+1)
        plt.imshow(X[i].squeeze(), cmap='gray')
        plt.title("COVID" if y[i] == 1 else "Normal")
        plt.axis('off')
    plt.suptitle(title)
    plt.show()

show_preprocessed_samples(X_train, y_train, "Sample Preprocessed Images")


In [ ]:
def plot_history(history, model_name):
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs = range(1, len(acc)+1)

    plt.figure(figsize=(14,5))

    plt.subplot(1,2,1)
    plt.plot(epochs, acc, 'bo-', label='Train Acc')
    plt.plot(epochs, val_acc, 'ro-', label='Val Acc')
    plt.title(f'{model_name} Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.subplot(1,2,2)
    plt.plot(epochs, loss, 'bo-', label='Train Loss')
    plt.plot(epochs, val_loss, 'ro-', label='Val Loss')
    plt.title(f'{model_name} Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout()
    plt.show()

# Plot all histories
for model_name, history in histories.items():
    plot_history(history, model_name)


In [ ]:
model_accuracies = {}

for name, model in models_list.items():
    if name == "Custom_CNN":
        _, acc = model.evaluate(X_test, y_test_cat, verbose=0)
    else:
        _, acc = model.evaluate(X_test_3c, y_test_cat, verbose=0)
    model_accuracies[name] = acc

# Plot
plt.figure(figsize=(10,6))
plt.bar(model_accuracies.keys(), model_accuracies.values(), color='skyblue')
plt.ylabel("Test Accuracy")
plt.title("Model Accuracy Comparison")
plt.xticks(rotation=45)
plt.ylim(0, 1)
plt.grid(True)
plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

def evaluate_model(name, model, X_test, y_test):
    print(f"\nEvaluating {name}...")
    y_pred_probs = model.predict(X_test)
    y_pred = np.argmax(y_pred_probs, axis=1)
    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=categories, yticklabels=categories)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(f'{name} Confusion Matrix')
    plt.show()

    print(classification_report(y_test, y_pred, target_names=categories))

# Run for all models
for name, model in models_list.items():
    if name == "Custom_CNN":
        evaluate_model(name, model, X_test, y_test)
    else:
        evaluate_model(name, model, X_test_3c, y_test)


In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.cm as cm

def make_gradcam_heatmap(img_array, model, last_conv_layer_name):
    # Build a model that maps the input image to the activations and predictions
    grad_model = tf.keras.models.Model(
        [model.inputs],
        [model.get_layer(last_conv_layer_name).output, model.output]
    )

    # Record operations for automatic differentiation
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        pred_index = tf.argmax(predictions[0])
        class_output = predictions[:, pred_index]

    # Compute the gradient of the top predicted class with regard to the output feature map
    grads = tape.gradient(class_output, conv_outputs)

    # Mean of the gradients over all channels
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # Weight the output feature map with the gradients
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    # Normalize the heatmap
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()


In [ ]:
import cv2

def superimpose_heatmap(heatmap, original_img, alpha=0.4):
    # Resize heatmap to image size
    heatmap = cv2.resize(heatmap, (original_img.shape[1], original_img.shape[0]))

    # Apply colormap
    heatmap_colored = cm.jet(heatmap)[..., :3]  # Remove alpha channel

    # Convert grayscale to RGB
    if len(original_img.shape) == 2:
        original_img = np.stack([original_img]*3, axis=-1)

    # Superimpose heatmap
    superimposed = heatmap_colored * alpha + original_img
    superimposed = np.clip(superimposed, 0, 1)
    return superimposed


In [ ]:
# Load one test image
i = 0  # index of test image
x_img = X_test_3c[i:i+1]  # shape: (1, 224, 224, 3)
original = X_test[i]      # grayscale image for overlay

# Choose the model (example: ResNet50)
model_to_explain = models_list["ResNet50"]
last_conv_layer = "conv5_block3_out"  # layer name for ResNet50

# Generate heatmap
heatmap = make_gradcam_heatmap(x_img, model_to_explain, last_conv_layer_name=last_conv_layer)

# Superimpose
superimposed = superimpose_heatmap(heatmap, original)

# Display original, heatmap, and overlay
import matplotlib.pyplot as plt

plt.figure(figsize=(15,5))

plt.subplot(1, 3, 1)
plt.title("Original X-ray")
plt.imshow(original, cmap='gray')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.title("Grad-CAM Heatmap")
plt.imshow(heatmap, cmap='jet')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.title("Superimposed")
plt.imshow(superimposed)
plt.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
import tensorflow as tf
import matplotlib.cm as cm

def make_gradcam_heatmap(img_array, model, last_conv_layer_name="conv5_block3_out"):
    grad_model = tf.keras.models.Model(
        [model.inputs], [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    # Gradient of the output neuron (class) w.r.t. feature map
    grads = tape.gradient(class_channel, conv_outputs)

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()


In [ ]:
def superimpose_heatmap(heatmap, original_img, alpha=0.4):
    # Resize heatmap to original image size
    heatmap = cv2.resize(heatmap, (original_img.shape[1], original_img.shape[0]))

    # Apply colormap
    heatmap_colored = cm.jet(heatmap)[..., :3]  # Remove alpha channel

    # Convert original image to 3 channels if it's grayscale
    if len(original_img.shape) == 2:
        original_img = np.stack([original_img]*3, axis=-1)

    superimposed_img = heatmap_colored * alpha + original_img
    superimposed_img = np.clip(superimposed_img, 0, 1)
    return superimposed_img


In [ ]:
# Load one test image
i = 0
x_img = X_test_3c[i:i+1]  # shape: (1, 224, 224, 3)
original = X_test[i]      # grayscale original (224, 224)

# Choose the model to explain (e.g., ResNet50)
model_to_explain = models_list["ResNet50"]
last_conv_layer = "conv5_block3_out"  # for ResNet50

# Generate heatmap
heatmap = make_gradcam_heatmap(x_img, model_to_explain, last_conv_layer_name=last_conv_layer)

# Superimpose
superimposed = superimpose_heatmap(heatmap, original)

# Show all
plt.figure(figsize=(15,5))
plt.subplot(1,3,1)
plt.title("Original X-ray")
plt.imshow(original, cmap='gray')
plt.axis('off')

plt.subplot(1,3,2)
plt.title("Grad-CAM Heatmap")
plt.imshow(heatmap, cmap='jet')
plt.axis('off')

plt.subplot(1,3,3)
plt.title("Superimposed")
plt.imshow(superimposed)
plt.axis('off')

plt.tight_layout()
plt.show()
